# Adapter Training — Chittagong Dialect → English

This notebook trains **LoRA adapters** on top of the dialect-tagged NLLB-200-600M model for the Chittagong dialect specifically, then evaluates on a held-out validation set using:

- **SacreBLEU** (corpus-level)
- **chrF++**
- **ROUGE-1, ROUGE-2, ROUGE-L**
- **METEOR**
- **Word Error Rate (WER)**
- **Character Error Rate (CER)**

### Key design decisions
- Only the **LoRA adapter weights** are trained — the base NLLB model is fully frozen
- Chittagong text gets **punctuation/whitespace-only normalization** (no full csebuet normalize)
- A small Standard Bangla anchor sample is mixed in to prevent catastrophic forgetting
- Adapter weights are saved separately — the base model is never modified

### Kaggle setup
- Accelerator: **GPU T4 x2**
- Internet: **On** (to install packages)
- Add your dialect-tagged model dataset: `/kaggle/input/nllb-dialect-model/`
- Add your Chittagong data dataset

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              ⚠️  AUTO-RUN GUARD  ⚠️                          ║
# ║  Delete or skip this cell, then run cells one at a time.    ║
# ╚══════════════════════════════════════════════════════════════╝
import sys
print("⛔ Auto-run blocked. Delete this cell and run manually with Shift+Enter.")
sys.exit("Auto-run guard triggered.")

In [4]:
!pip install transformers==5.2.0 peft==0.18.1 datasets sacrebleu \
             rouge-score nltk jiwer sentencepiece accelerate \
             evaluate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.0 MB/s eta 0:00:00a 0:00:01


In [5]:
import json

config_path = "/kaggle/input/datasets/farhanaadri/nllb-dialect-model/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "M2M100ForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "dtype": "float32",
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "max_position_embeddings": 1024,
  "model_type": "m2m_100",
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "scale_embedding": true,
  "tie_word_embeddings": true,
  "tokenizer_class": "NllbTokenizer",
  "transformers_version": "5.2.0",
  "use_cache": true,
  "vocab_size": 256209
}


In [6]:
# ── Configuration — EDIT THESE ────────────────────────────────────────────────

# Path to your dialect-tagged NLLB model (from the add_dialect_tags notebook)
BASE_MODEL_PATH = "/kaggle/input/datasets/farhanaadri/nllb-dialect-model"

# Path to your Chittagong CSV data
# Expected columns: 'dialect_text' (Chittagong), 'english_text' (English translation)
CHITTAGONG_CSV  = "/kaggle/input/datasets/farhanaadri/all-dialects-data/dataset_translated/train/chittagong_dialect_final.csv"  # <-- CHANGE
DIALECT_COL     = "dialect_speech"    # <-- column with Chittagong text
ENGLISH_COL     = "English_translation"    # <-- column with English translation

# Optional: Standard Bangla CSV for anchor mixing (set to None to skip)
# Prevents catastrophic forgetting of standard Bangla during adapter training
BANGLA_CSV      = CHITTAGONG_CSV  # e.g. "/kaggle/input/your-data/standard_bangla.csv"
BANGLA_DIALECT_COL  = "Standard_Bangla"
BANGLA_ENGLISH_COL  = "English_translation"
BANGLA_ANCHOR_FRAC  = 0.10   # mix 10% standard Bangla into each training batch



# Train/validation split
DIALECT_COL_VAL     = "chittagong_bangla_speech "    # <-- column with Chittagong text
ENGLISH_COL_VAL     = "english_speech"    # <-- column with English translation
# VAL_CSV = "/kaggle/input/datasets/farhanaadri/all-dialects-data/dataset_translated/validation/chittagong_validation.csv"   # <-- ADD THIS
VAL_CSV = "/kaggle/input/datasets/farhanaadri/vashantor/Vashantor A Large-scale Multilingual Benchmark Dataset for Automated Translation of Bangla Regional Dialects to Bangla Language/Vashantor_CSV_Format/Vashantor_CSV_Format/Validation/Chittagong Validation Translation.csv"   # <-- ADD THIS

# LoRA adapter hyperparameters
LORA_R          = 16      # rank — higher = more expressive but more parameters
LORA_ALPHA      = 32      # scaling factor (usually 2x rank)
LORA_DROPOUT    = 0.05
# Which layers to apply LoRA to. These are the projection matrices in each
# transformer block. q_proj and v_proj is standard; add k_proj, out_proj
# for more capacity if your data is large enough.
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Training hyperparameters
NUM_EPOCHS      = 30
BATCH_SIZE      = 16      # per-device batch size
GRAD_ACCUM      = 2       # effective batch = BATCH_SIZE * GRAD_ACCUM * num_gpus
LEARNING_RATE   = 3e-4    # adapters can use higher LR than full fine-tuning
WARMUP_STEPS    = 100
MAX_SOURCE_LEN  = 128
MAX_TARGET_LEN  = 128
WEIGHT_DECAY    = 0.01
FP16            = True    # use mixed precision on T4
RANDOM_SEED = 42
# Generation hyperparameters (used during evaluation)
NUM_BEAMS       = 5

# Where to save adapter weights
OUTPUT_DIR      = "/kaggle/working/chittagong_adapter_v2"

# Dialect tag for Chittagong (must match what was set in add_dialect_tags notebook)
SRC_LANG        = "bng_chittagong"
TGT_LANG        = "eng_Latn"

print("Configuration loaded.")

Configuration loaded.


In [7]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

import torch
from torch.utils.data import Dataset

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from transformers import (
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel,
)
import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from jiwer import wer, cer

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"Device  : {device}")
print(f"Num GPUs: {n_gpus}")
if device == "cuda":
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Device  : cuda
Num GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [8]:
# ── Chittagong-safe normalization ─────────────────────────────────────────────
#
# Based on verification results: csebuet normalizer damages Chittagong-specific
# features (loanwords, nasalization, pronouns). We apply ONLY:
#   1. Whitespace normalization (collapse multiple spaces, strip)
#   2. Punctuation normalization (Unicode punctuation → ASCII equivalents)
#   3. Zero-width character removal
# We deliberately do NOT apply phoneme normalization or unicode composition
# that the full csebuet normalizer would do.

# Unicode punctuation → ASCII map
_PUNCT_MAP = str.maketrans({
    "\u201c": '"', "\u201d": '"',   # curly double quotes
    "\u2018": "'", "\u2019": "'",   # curly single quotes
    "\u2013": "-", "\u2014": "-",   # en-dash, em-dash
    "\u2026": "...",                 # ellipsis
    "\u00a0": " ",                   # non-breaking space
    "\u200b": "",                    # zero-width space
    "\u200c": "",                    # zero-width non-joiner
    "\u200d": "",                    # zero-width joiner
    "\ufeff": "",                    # BOM
})

def normalize_chittagong(text: str) -> str:
    """Safe normalization that preserves Chittagong dialectal features."""
    if not isinstance(text, str):
        return ""
    text = text.translate(_PUNCT_MAP)          # punctuation normalization
    text = re.sub(r"[\u200b-\u200f\ufeff]", "", text)  # zero-width chars
    text = re.sub(r" +", " ", text)            # collapse multiple spaces
    text = text.strip()
    return text

# Quick test
test = "আঁই\u200b হন্ডে  যাইচ\u201d"
print(f"Before: {repr(test)}")
print(f"After : {repr(normalize_chittagong(test))}")
print("Normalization function ready.")

Before: 'আঁই\u200b হন্ডে  যাইচ”'
After : 'আঁই হন্ডে যাইচ"'
Normalization function ready.


In [9]:
# ── Load and split data ───────────────────────────────────────────────────────

df = pd.read_csv(CHITTAGONG_CSV)
print(f"Loaded {len(df)} rows from {CHITTAGONG_CSV}")
print(f"Columns: {list(df.columns)}")

# Drop rows with missing values in either column
df = df[[DIALECT_COL, ENGLISH_COL]].dropna().reset_index(drop=True)
df = df[df[DIALECT_COL].str.strip().ne("") & df[ENGLISH_COL].str.strip().ne("")]
print(f"After dropping nulls/empty: {len(df)} rows")

# Apply safe normalization
df[DIALECT_COL] = df[DIALECT_COL].apply(normalize_chittagong)
df[ENGLISH_COL] = df[ENGLISH_COL].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Use full training CSV as train, load separate validation CSV
df_train = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

df_val_raw = pd.read_csv(VAL_CSV)
df_val = df_val_raw[[DIALECT_COL_VAL, ENGLISH_COL_VAL]].dropna().reset_index(drop=True)
df_val[DIALECT_COL_VAL] = df_val[DIALECT_COL_VAL].apply(normalize_chittagong)
df_val[ENGLISH_COL_VAL] = df_val[ENGLISH_COL_VAL].apply(lambda x: x.strip() if isinstance(x, str) else x)

print(f"\nTrain samples : {len(df_train)}")
print(f"Val samples   : {len(df_val)}")
print(f"\nSample training row:")
print(f"  Source : {df_train[DIALECT_COL].iloc[0]}")
print(f"  Target : {df_train[ENGLISH_COL].iloc[0]}")

Loaded 12957 rows from /kaggle/input/datasets/farhanaadri/all-dialects-data/dataset_translated/train/chittagong_dialect_final.csv
Columns: ['Standard_Bangla', 'dialect_speech', 'English_translation']
After dropping nulls/empty: 12957 rows

Train samples : 12957
Val samples   : 250

Sample training row:
  Source : আল্লা দান অর আত আরো লম্বা গরন
  Target : May Allah extend the hand of giving even further.


In [10]:
# ── Optionally load Standard Bangla anchor data ───────────────────────────────
#
# Mixing a small fraction of standard Bangla samples prevents the adapter from
# overfitting so hard to Chittagong that it damages the model's general Bangla
# translation ability.

df_bangla_train = None

if BANGLA_CSV is not None:
    df_bn = pd.read_csv(BANGLA_CSV)
    df_bn = df_bn[[BANGLA_DIALECT_COL, BANGLA_ENGLISH_COL]].dropna().reset_index(drop=True)
    df_bn.columns = [DIALECT_COL, ENGLISH_COL]  # rename to match

    # Take anchor_frac proportion relative to Chittagong training size
    n_anchor = int(len(df_train) * BANGLA_ANCHOR_FRAC)
    df_bangla_train = df_bn.sample(
        n=min(n_anchor, len(df_bn)), random_state=RANDOM_SEED
    ).reset_index(drop=True)

    print(f"Standard Bangla anchor samples: {len(df_bangla_train)}")
else:
    print("No Bangla anchor data configured (BANGLA_CSV is None). Skipping.")

Standard Bangla anchor samples: 1295


In [11]:
# ── Load tokenizer and model ──────────────────────────────────────────────────

print(f"Loading tokenizer from: {BASE_MODEL_PATH}")
tokenizer = NllbTokenizer.from_pretrained(BASE_MODEL_PATH, use_fast=False)
print(f"  Vocab size: {len(tokenizer)}")

# Verify our dialect tag is present
src_tag_id = tokenizer.convert_tokens_to_ids(SRC_LANG)
assert src_tag_id != tokenizer.unk_token_id, (
    f"Source lang tag '{SRC_LANG}' not found in tokenizer! "
    f"Make sure you're loading from the dialect-tagged model, not the original NLLB."
)
tgt_tag_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f"  {SRC_LANG} → token_id {src_tag_id} ✅")
print(f"  {TGT_LANG} → token_id {tgt_tag_id} ✅")

print(f"\nLoading model from: {BASE_MODEL_PATH}")
from transformers import AutoConfig

# Load and fix config before loading model
config = AutoConfig.from_pretrained(BASE_MODEL_PATH)

# Fix: the tied_weights_keys field may be a list instead of dict in saved config
if hasattr(config, 'tied_weights_keys') and isinstance(config.tied_weights_keys, list):
    pass  # this is fine, just suppress

model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_PATH,
    config=config,
    dtype=torch.float16 if FP16 else torch.float32,
    ignore_mismatched_sizes=True,
)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer from: /kaggle/input/datasets/farhanaadri/nllb-dialect-model
  Vocab size: 256209
  bng_chittagong → token_id 256207 ✅
  eng_Latn → token_id 256047 ✅

Loading model from: /kaggle/input/datasets/farhanaadri/nllb-dialect-model


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Parameters: 615,076,864


In [12]:
import transformers
import peft
import torch

print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"torch        : {torch.__version__}")
print(f"torch cuda   : {torch.cuda.is_available()}")

transformers : 5.2.0
peft         : 0.18.1
torch        : 2.9.0+cu126
torch cuda   : True


In [13]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets/farhanaadri/nllb-dialect-model"):
    for f in files:
        print(os.path.join(root, f))


/kaggle/input/datasets/farhanaadri/nllb-dialect-model/config.json
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/tokenizer.json
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/tokenizer_config.json
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/model.safetensors
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/sentencepiece.bpe.model
/kaggle/input/datasets/farhanaadri/nllb-dialect-model/generation_config.json


In [14]:
# ── Hyperparameter Search Grid ────────────────────────────────────────────────
#
# We run 5-epoch trials for each config, comparing:
#   Phase 1 — Learning Rate       : 1e-4, 3e-4, 5e-4
#   Phase 2 — LR Scheduler        : linear, cosine, cosine_with_restarts
#   Phase 3 — LoRA Target Modules : [q,v] vs [q,v,k,out]
#
# After all training phases, we test NUM_BEAMS and length_penalty
# on the best trained model (inference-only, no retraining).
#
# All results are collected in HP_RESULTS and printed in a final summary table.

HP_RESULTS = []   # list of dicts, one per trial

# Shared fixed values across all training trials
HP_NUM_EPOCHS   = 5
HP_BATCH_SIZE   = BATCH_SIZE      # 16
HP_GRAD_ACCUM   = GRAD_ACCUM      # 2
HP_WARMUP_STEPS = WARMUP_STEPS    # 100
HP_WEIGHT_DECAY = WEIGHT_DECAY    # 0.01

# Phase 1 grid — vary LR only, fix scheduler=linear, modules=[q,v]
PHASE1_CONFIGS = [
    {"lr": 1e-4, "scheduler": "linear",  "modules": ["q_proj", "v_proj"], "label": "LR=1e-4"},
    {"lr": 3e-4, "scheduler": "linear",  "modules": ["q_proj", "v_proj"], "label": "LR=3e-4 (baseline)"},
    {"lr": 5e-4, "scheduler": "linear",  "modules": ["q_proj", "v_proj"], "label": "LR=5e-4"},
]

# Phase 2 grid — fix best LR from phase1 (set below after phase1), vary scheduler
# We pre-define all 3 scheduler options; best LR will be filled in at runtime
PHASE2_SCHEDULERS = ["linear", "cosine", "cosine_with_restarts"]

# Phase 3 grid — fix best LR + scheduler from phases 1-2, vary target modules
PHASE3_MODULE_OPTIONS = [
    {"modules": ["q_proj", "v_proj"],                       "label": "modules=[q,v]"},
    {"modules": ["q_proj", "v_proj", "k_proj", "out_proj"], "label": "modules=[q,v,k,out]"},
]

# Inference-only grid — beam sizes and length penalties tested on best trained model
INFERENCE_BEAM_OPTIONS    = [3, 5, 8]
INFERENCE_LENPEN_OPTIONS  = [0.8, 1.0, 1.2]

print("Hyperparameter search grid defined.")
print(f"  Phase 1: {len(PHASE1_CONFIGS)} LR trials  x  {HP_NUM_EPOCHS} epochs each")
print(f"  Phase 2: {len(PHASE2_SCHEDULERS)} scheduler trials  x  {HP_NUM_EPOCHS} epochs each")
print(f"  Phase 3: {len(PHASE3_MODULE_OPTIONS)} module trials  x  {HP_NUM_EPOCHS} epochs each")
print(f"  Inference grid: {len(INFERENCE_BEAM_OPTIONS)} beam values x {len(INFERENCE_LENPEN_OPTIONS)} length penalties")


Hyperparameter search grid defined.
  Phase 1: 3 LR trials  x  5 epochs each
  Phase 2: 3 scheduler trials  x  5 epochs each
  Phase 3: 2 module trials  x  5 epochs each
  Inference grid: 3 beam values x 3 length penalties


In [15]:
# ── Dataset class ─────────────────────────────────────────────────────────────

class DialectDataset(Dataset):
    """
    PyTorch Dataset for dialect → English translation.
    Handles tokenization with the correct source language tag per sample.
    """
    def __init__(
        self,
        sources: list,
        targets: list,
        tokenizer: NllbTokenizer,
        src_lang: str,
        tgt_lang: str,
        max_source_len: int = 128,
        max_target_len: int = 128,
    ):
        self.sources        = sources
        self.targets        = targets
        self.tokenizer      = tokenizer
        self.src_lang       = src_lang
        self.tgt_lang       = tgt_lang
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        source = str(self.sources[idx])
        target = str(self.targets[idx])

        # Tokenize source with dialect tag
        self.tokenizer.src_lang = self.src_lang
        model_inputs = self.tokenizer(
            source,
            max_length=self.max_source_len,
            truncation=True,
            padding=False,
        )

        # Tokenize target (English)
        self.tokenizer.src_lang = self.tgt_lang
        labels = self.tokenizer(
            target,
            max_length=self.max_target_len,
            truncation=True,
            padding=False,
        )
        # Reset src_lang back to source language
        self.tokenizer.src_lang = self.src_lang

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


# ── Build datasets (identical to original) ────────────────────────────────────

train_sources  = df_train[DIALECT_COL].tolist()
train_targets  = df_train[ENGLISH_COL].tolist()
train_src_langs = [SRC_LANG] * len(train_sources)

if df_bangla_train is not None:
    train_sources   += df_bangla_train[DIALECT_COL].tolist()
    train_targets   += df_bangla_train[ENGLISH_COL].tolist()
    train_src_langs += ["ben_Beng"] * len(df_bangla_train)
    print(f"Combined training set: {len(train_sources)} samples")
    print(f"  ({len(df_train)} Chittagong + {len(df_bangla_train)} Standard Bangla anchor)")

import random
combined = list(zip(train_sources, train_targets, train_src_langs))
random.seed(RANDOM_SEED)
random.shuffle(combined)
train_sources, train_targets, train_src_langs = zip(*combined)

train_dataset = DialectDataset(
    list(train_sources), list(train_targets),
    tokenizer, SRC_LANG, TGT_LANG,
    MAX_SOURCE_LEN, MAX_TARGET_LEN
)

val_dataset = DialectDataset(
    df_val[DIALECT_COL_VAL].tolist(), df_val[ENGLISH_COL_VAL].tolist(),
    tokenizer, SRC_LANG, TGT_LANG,
    MAX_SOURCE_LEN, MAX_TARGET_LEN
)

print(f"\nTrain dataset size : {len(train_dataset)}")
print(f"Val dataset size   : {len(val_dataset)}")


Combined training set: 14252 samples
  (12957 Chittagong + 1295 Standard Bangla anchor)

Train dataset size : 14252
Val dataset size   : 250


In [16]:
# ── Helper: run one training trial ───────────────────────────────────────────
#
# Loads a fresh LoRA model each call so weights never bleed between trials.
# Returns (trainer, trained_model, final_eval_loss).

import copy
import time

def run_training_trial(
    trial_label: str,
    lr: float,
    scheduler_type: str,
    target_modules: list,
    num_epochs: int = HP_NUM_EPOCHS,
    batch_size: int = HP_BATCH_SIZE,
    grad_accum: int = HP_GRAD_ACCUM,
    warmup_steps: int = HP_WARMUP_STEPS,
    weight_decay: float = HP_WEIGHT_DECAY,
):
    """Train a fresh LoRA adapter with the given config for num_epochs epochs."""
    print(f"\n{'='*65}")
    print(f"  TRIAL: {trial_label}")
    print(f"  lr={lr}  scheduler={scheduler_type}")
    print(f"  target_modules={target_modules}")
    print(f"  epochs={num_epochs}  batch={batch_size}  grad_accum={grad_accum}")
    print(f"{'='*65}")

    # ── Fresh base model load ──────────────────────────────────────────────────
    from transformers import AutoConfig, AutoModelForSeq2SeqLM
    from peft import get_peft_model, LoraConfig, TaskType

    trial_config = AutoConfig.from_pretrained(BASE_MODEL_PATH)
    trial_model  = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL_PATH,
        config=trial_config,
        dtype=torch.float16 if FP16 else torch.float32,
        ignore_mismatched_sizes=True,
    )

    lora_cfg = LoraConfig(
        task_type      = TaskType.SEQ_2_SEQ_LM,
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = target_modules,
        bias           = "none",
    )
    trial_model = get_peft_model(trial_model, lora_cfg)

    trainable = sum(p.numel() for p in trial_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trial_model.parameters())
    print(f"  Trainable params: {trainable:,}  ({100*trainable/total:.3f}%)")

    # ── Training args ──────────────────────────────────────────────────────────
    trial_output_dir = f"/kaggle/working/hp_trial_{trial_label.replace(' ', '_').replace('=', '').replace(',','')}"

    trial_args = Seq2SeqTrainingArguments(
        output_dir                  = trial_output_dir,
        num_train_epochs            = num_epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        gradient_accumulation_steps = grad_accum,
        learning_rate               = lr,
        warmup_steps                = warmup_steps,
        weight_decay                = weight_decay,
        lr_scheduler_type           = scheduler_type,
        fp16                        = FP16,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        predict_with_generate       = False,   # faster during search; we eval manually after
        logging_steps               = 50,
        save_total_limit            = 1,
        report_to                   = "none",
        dataloader_num_workers      = 2,
        ddp_find_unused_parameters  = False,
    )

    data_collator_trial = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=trial_model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8 if FP16 else None,
    )

    trainer = Seq2SeqTrainer(
        model            = trial_model,
        args             = trial_args,
        train_dataset    = train_dataset,
        eval_dataset     = val_dataset,
        processing_class = tokenizer,
        data_collator    = data_collator_trial,
        callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    t0 = time.time()
    train_result = trainer.train()
    elapsed = time.time() - t0

    # Pull best eval loss from trainer state
    best_eval_loss = trainer.state.best_metric
    if best_eval_loss is None:
        # fallback: last eval loss
        logs = [l for l in trainer.state.log_history if "eval_loss" in l]
        best_eval_loss = logs[-1]["eval_loss"] if logs else float("nan")

    print(f"\n  ✓ Done in {elapsed:.0f}s")
    print(f"  train_loss   : {train_result.training_loss:.4f}")
    print(f"  best_eval_loss: {best_eval_loss:.4f}")
    print(f"  steps run    : {train_result.global_step}")

    return trainer, trial_model, best_eval_loss, train_result.training_loss, elapsed


# ── Helper: run inference + full metrics on a model ───────────────────────────

def run_inference_eval(
    eval_model,
    num_beams: int   = NUM_BEAMS,
    length_penalty: float = 1.0,
    inference_batch: int  = 16,
    label: str = "",
):
    """
    Run beam-search generation on val set and compute all metrics.
    Returns a dict of metric scores.
    """
    import sacrebleu as sb
    from rouge_score import rouge_scorer as rs_mod
    from nltk.translate.meteor_score import meteor_score as ms_fn
    from jiwer import wer as wer_fn, cer as cer_fn

    eval_model.eval()
    eng_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
    val_sources  = df_val[DIALECT_COL_VAL].tolist()
    val_refs     = df_val[ENGLISH_COL_VAL].tolist()
    tokenizer.src_lang = SRC_LANG
    predictions = []

    for i in range(0, len(val_sources), inference_batch):
        batch_texts = val_sources[i : i + inference_batch]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        ).to(device)
        with torch.no_grad():
            generated = eval_model.generate(
                **inputs,
                forced_bos_token_id=eng_token_id,
                max_length=MAX_TARGET_LEN,
                num_beams=num_beams,
                length_penalty=length_penalty,
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        predictions.extend(decoded)

    refs_clean  = [r.strip() for r in val_refs]
    preds_clean = [p.strip() for p in predictions]

    # SacreBLEU
    bleu  = sb.corpus_bleu(preds_clean, [refs_clean]).score
    # chrF++
    chrf  = sb.corpus_chrf(preds_clean, [refs_clean], word_order=2).score
    # ROUGE
    scorer_r = rs_mod.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=False)
    r1, r2, rL = [], [], []
    for p, r in zip(preds_clean, refs_clean):
        sc = scorer_r.score(r, p)
        r1.append(sc["rouge1"].fmeasure)
        r2.append(sc["rouge2"].fmeasure)
        rL.append(sc["rougeL"].fmeasure)
    rouge1 = np.mean(r1) * 100
    rouge2 = np.mean(r2) * 100
    rougeL = np.mean(rL) * 100
    # METEOR
    met_scores = []
    for p, r in zip(preds_clean, refs_clean):
        pt, rt = p.split(), r.split()
        met_scores.append(ms_fn([rt], pt) if pt and rt else 0.0)
    meteor = np.mean(met_scores) * 100
    # WER / CER
    wer_score = wer_fn(refs_clean, preds_clean) * 100
    cer_score = cer_fn(refs_clean, preds_clean) * 100

    result = {
        "label"       : label,
        "num_beams"   : num_beams,
        "length_pen"  : length_penalty,
        "sacrebleu"   : round(bleu,  4),
        "chrf"        : round(chrf,  4),
        "rouge1"      : round(rouge1,4),
        "rouge2"      : round(rouge2,4),
        "rougeL"      : round(rougeL,4),
        "meteor"      : round(meteor, 4),
        "wer"         : round(wer_score, 4),
        "cer"         : round(cer_score, 4),
    }

    print(f"  [{label}]  BLEU={bleu:.2f}  chrF={chrf:.2f}  ROUGE-1={rouge1:.2f}  "
          f"METEOR={meteor:.2f}  WER={wer_score:.2f}%")
    return result


print("Helper functions ready.")


Helper functions ready.


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — Learning Rate Sweep
# Fixed: scheduler=linear, modules=[q_proj, v_proj]
# Varying: lr ∈ {1e-4, 3e-4, 5e-4}
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "█"*65)
print("  PHASE 1: LEARNING RATE SWEEP")
print("█"*65)

phase1_results = []

for cfg in PHASE1_CONFIGS:
    trainer, trained_model, best_eval_loss, train_loss, elapsed = run_training_trial(
        trial_label    = cfg["label"],
        lr             = cfg["lr"],
        scheduler_type = cfg["scheduler"],
        target_modules = cfg["modules"],
    )

    # Run inference eval with default beams/length_penalty
    metrics = run_inference_eval(
        trained_model,
        num_beams      = NUM_BEAMS,
        length_penalty = 1.0,
        label          = cfg["label"],
    )

    row = {
        "phase"          : 1,
        "label"          : cfg["label"],
        "lr"             : cfg["lr"],
        "scheduler"      : cfg["scheduler"],
        "modules"        : str(cfg["modules"]),
        "train_loss"     : round(train_loss, 4),
        "best_eval_loss" : round(best_eval_loss, 4),
        "elapsed_s"      : round(elapsed, 1),
        **metrics,
    }
    phase1_results.append(row)
    HP_RESULTS.append(row)

    # Free GPU memory before next trial
    del trained_model, trainer
    torch.cuda.empty_cache()

# ── Phase 1 summary ──────────────────────────────────────────────────────────
print("\n" + "─"*65)
print("  PHASE 1 SUMMARY")
print("─"*65)
print(f"  {'Label':<22} {'LR':<8} {'EvalLoss':<10} {'BLEU':<8} {'chrF':<8} {'METEOR':<8} {'WER%':<8}")
print("  " + "-"*62)
for r in phase1_results:
    print(f"  {r['label']:<22} {r['lr']:<8} {r['best_eval_loss']:<10.4f} "
          f"{r['sacrebleu']:<8.2f} {r['chrf']:<8.2f} {r['meteor']:<8.2f} {r['wer']:<8.2f}")

# Pick best LR by lowest eval_loss
best_phase1 = min(phase1_results, key=lambda x: x["best_eval_loss"])
BEST_LR = best_phase1["lr"]
print(f"\n  ✓ Best LR for Phase 2: {BEST_LR}  (eval_loss={best_phase1['best_eval_loss']:.4f})")



█████████████████████████████████████████████████████████████████
  PHASE 1: LEARNING RATE SWEEP
█████████████████████████████████████████████████████████████████

  TRIAL: LR=1e-4
  lr=0.0001  scheduler=linear
  target_modules=['q_proj', 'v_proj']
  epochs=5  batch=16  grad_accum=2


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable params: 2,359,296  (0.382%)


Epoch,Training Loss,Validation Loss
1,9.866563,4.019531
2,9.166562,3.640625
3,8.794531,3.482422
4,8.718516,3.384766
5,8.521445,3.376953



  ✓ Done in 1472s
  train_loss   : 9.1779
  best_eval_loss: 3.3770
  steps run    : 1115
  [LR=1e-4]  BLEU=11.31  chrF=32.91  ROUGE-1=42.19  METEOR=32.16  WER=84.12%

  TRIAL: LR=3e-4 (baseline)
  lr=0.0003  scheduler=linear
  target_modules=['q_proj', 'v_proj']
  epochs=5  batch=16  grad_accum=2


...


  ✓ Done in 1492s
  train_loss   : 8.3402
  best_eval_loss: 2.8281
  steps run    : 1115
  [LR=3e-4 (baseline)]  BLEU=17.33  chrF=40.56  ROUGE-1=51.17  METEOR=39.69  WER=74.41%

  TRIAL: LR=5e-4
  lr=0.0005  scheduler=linear
  target_modules=['q_proj', 'v_proj']
  epochs=5  batch=16  grad_accum=2


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable params: 2,359,296  (0.382%)


Epoch,Training Loss,Validation Loss
1,8.852344,3.349609
2,8.057031,2.873047
3,7.442812,2.697266
4,7.304414,2.542969
5,6.986641,2.500000



  ✓ Done in 1478s
  train_loss   : 7.9181
  best_eval_loss: 2.5000
  steps run    : 1115
  [LR=5e-4]  BLEU=20.90  chrF=46.25  ROUGE-1=56.90  METEOR=45.73  WER=71.65%

─────────────────────────────────────────────────────────────────
  PHASE 1 SUMMARY
─────────────────────────────────────────────────────────────────
  Label                  LR       EvalLoss   BLEU     chrF     METEOR   WER%    
  --------------------------------------------------------------
  LR=1e-4                0.0001   3.3770     11.31    32.91    32.16    84.12   
  LR=3e-4 (baseline)     0.0003   2.8281     17.33    40.56    39.69    74.41   
  LR=5e-4                0.0005   2.5000     20.90    46.25    45.73    71.65   

  ✓ Best LR for Phase 2: 0.0005  (eval_loss=2.5000)


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — LR Scheduler Sweep
# Fixed: lr=BEST_LR (from Phase 1), modules=[q_proj, v_proj]
# Varying: scheduler ∈ {linear, cosine, cosine_with_restarts}
# Note: linear is already run in Phase 1 — we reuse that result, only run 2 new.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "█"*65)
print(f"  PHASE 2: SCHEDULER SWEEP  (lr={BEST_LR})")
print("█"*65)

phase2_results = []

# Carry over the linear result from Phase 1 for the best LR
linear_baseline = next(r for r in phase1_results if r["lr"] == BEST_LR and r["scheduler"] == "linear")
phase2_results.append(linear_baseline)
print(f"  [Reusing Phase 1 result for scheduler=linear, lr={BEST_LR}]")

for sched in ["cosine", "cosine_with_restarts"]:
    label = f"LR={BEST_LR}_{sched}"
    trainer, trained_model, best_eval_loss, train_loss, elapsed = run_training_trial(
        trial_label    = label,
        lr             = BEST_LR,
        scheduler_type = sched,
        target_modules = ["q_proj", "v_proj"],
    )

    metrics = run_inference_eval(
        trained_model,
        num_beams      = NUM_BEAMS,
        length_penalty = 1.0,
        label          = label,
    )

    row = {
        "phase"          : 2,
        "label"          : label,
        "lr"             : BEST_LR,
        "scheduler"      : sched,
        "modules"        : str(["q_proj", "v_proj"]),
        "train_loss"     : round(train_loss, 4),
        "best_eval_loss" : round(best_eval_loss, 4),
        "elapsed_s"      : round(elapsed, 1),
        **metrics,
    }
    phase2_results.append(row)
    HP_RESULTS.append(row)

    del trained_model, trainer
    torch.cuda.empty_cache()

# ── Phase 2 summary ───────────────────────────────────────────────────────────
print("\n" + "─"*65)
print("  PHASE 2 SUMMARY")
print("─"*65)
print(f"  {'Label':<30} {'Scheduler':<25} {'EvalLoss':<10} {'BLEU':<8} {'chrF':<8}")
print("  " + "-"*62)
for r in phase2_results:
    print(f"  {r['label']:<30} {r['scheduler']:<25} {r['best_eval_loss']:<10.4f} "
          f"{r['sacrebleu']:<8.2f} {r['chrf']:<8.2f}")

best_phase2  = min(phase2_results, key=lambda x: x["best_eval_loss"])
BEST_SCHEDULER = best_phase2["scheduler"]
print(f"\n  ✓ Best scheduler for Phase 3: {BEST_SCHEDULER}  (eval_loss={best_phase2['best_eval_loss']:.4f})")



█████████████████████████████████████████████████████████████████
  PHASE 2: SCHEDULER SWEEP  (lr=0.0005)
█████████████████████████████████████████████████████████████████
  [Reusing Phase 1 result for scheduler=linear, lr=0.0005]

  TRIAL: LR=0.0005_cosine
  lr=0.0005  scheduler=cosine
  target_modules=['q_proj', 'v_proj']
  epochs=5  batch=16  grad_accum=2


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable params: 2,359,296  (0.382%)


Epoch,Training Loss,Validation Loss
1,8.841719,3.324219
2,8.009297,2.806641
3,7.353711,2.626953
4,7.222891,2.496094
5,6.956367,2.496094



  ✓ Done in 1471s
  train_loss   : 7.8648
  best_eval_loss: 2.4961
  steps run    : 1115
  [LR=0.0005_cosine]  BLEU=24.21  chrF=45.79  ROUGE-1=56.28  METEOR=44.74  WER=68.44%

  TRIAL: LR=0.0005_cosine_with_restarts
  lr=0.0005  scheduler=cosine_with_restarts
  target_modules=['q_proj', 'v_proj']
  epochs=5  batch=16  grad_accum=2


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable params: 2,359,296  (0.382%)


Epoch,Training Loss,Validation Loss
1,8.845352,3.326172
2,8.008437,2.816406
3,7.357500,2.632812
4,7.226562,2.507812
5,6.956875,2.503906



  ✓ Done in 1465s
  train_loss   : 7.8668
  best_eval_loss: 2.5039
  steps run    : 1115
  [LR=0.0005_cosine_with_restarts]  BLEU=25.22  chrF=46.94  ROUGE-1=57.30  METEOR=45.95  WER=68.31%

─────────────────────────────────────────────────────────────────
  PHASE 2 SUMMARY
─────────────────────────────────────────────────────────────────
  Label                          Scheduler                 EvalLoss   BLEU     chrF    
  --------------------------------------------------------------
  LR=5e-4                        linear                    2.5000     20.90    46.25   
  LR=0.0005_cosine               cosine                    2.4961     24.21    45.79   
  LR=0.0005_cosine_with_restarts cosine_with_restarts      2.5039     25.22    46.94   

  ✓ Best scheduler for Phase 3: cosine  (eval_loss=2.4961)


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3 — LoRA Target Modules Sweep
# Fixed: lr=BEST_LR, scheduler=BEST_SCHEDULER
# Varying: modules ∈ {[q,v], [q,v,k,out]}
# NOTE: [q,v,k,out] uses more VRAM. If OOM occurs, the except block
#       catches it, logs the OOM, and skips gracefully.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "█"*65)
print(f"  PHASE 3: TARGET MODULES SWEEP  (lr={BEST_LR}, scheduler={BEST_SCHEDULER})")
print("█"*65)

phase3_results = []

# Carry over [q,v] result from Phase 2
qv_baseline = next(r for r in phase2_results if r["scheduler"] == BEST_SCHEDULER)
phase3_results.append(qv_baseline)
print(f"  [Reusing Phase 2 result for modules=[q,v]]")

for opt in PHASE3_MODULE_OPTIONS[1:]:   # skip [q,v] (already have it)
    label = f"LR={BEST_LR}_{BEST_SCHEDULER}_{opt['label']}"
    try:
        trainer, trained_model, best_eval_loss, train_loss, elapsed = run_training_trial(
            trial_label    = label,
            lr             = BEST_LR,
            scheduler_type = BEST_SCHEDULER,
            target_modules = opt["modules"],
        )

        metrics = run_inference_eval(
            trained_model,
            num_beams      = NUM_BEAMS,
            length_penalty = 1.0,
            label          = label,
        )

        row = {
            "phase"          : 3,
            "label"          : label,
            "lr"             : BEST_LR,
            "scheduler"      : BEST_SCHEDULER,
            "modules"        : str(opt["modules"]),
            "train_loss"     : round(train_loss, 4),
            "best_eval_loss" : round(best_eval_loss, 4),
            "elapsed_s"      : round(elapsed, 1),
            **metrics,
        }
        phase3_results.append(row)
        HP_RESULTS.append(row)

        del trained_model, trainer
        torch.cuda.empty_cache()

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\n  ⚠️  OOM on modules={opt['modules']} — skipping this trial.")
            print(f"       To retry: set BATCH_SIZE=8 and GRAD_ACCUM=4 at the top.")
            torch.cuda.empty_cache()
        else:
            raise

# ── Phase 3 summary ───────────────────────────────────────────────────────────
print("\n" + "─"*65)
print("  PHASE 3 SUMMARY")
print("─"*65)
print(f"  {'Label':<20} {'Modules':<35} {'EvalLoss':<10} {'BLEU':<8} {'chrF':<8}")
print("  " + "-"*62)
for r in phase3_results:
    print(f"  {r['label'][:20]:<20} {r['modules']:<35} {r['best_eval_loss']:<10.4f} "
          f"{r['sacrebleu']:<8.2f} {r['chrf']:<8.2f}")

best_phase3    = min(phase3_results, key=lambda x: x["best_eval_loss"])
BEST_MODULES   = eval(best_phase3["modules"])   # convert string back to list
print(f"\n  ✓ Best modules for final model: {BEST_MODULES}")
print(f"\n  ── BEST OVERALL TRAINING CONFIG ─────────────────────────────")
print(f"     lr             = {BEST_LR}")
print(f"     scheduler      = {BEST_SCHEDULER}")
print(f"     target_modules = {BEST_MODULES}")
print(f"  ──────────────────────────────────────────────────────────────")



█████████████████████████████████████████████████████████████████
  PHASE 3: TARGET MODULES SWEEP  (lr=0.0005, scheduler=cosine)
█████████████████████████████████████████████████████████████████
  [Reusing Phase 2 result for modules=[q,v]]

  TRIAL: LR=0.0005_cosine_modules=[q,v,k,out]
  lr=0.0005  scheduler=cosine
  target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj']
  epochs=5  batch=16  grad_accum=2


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable params: 4,718,592  (0.761%)


Epoch,Training Loss,Validation Loss
1,8.514180,3.099609
2,7.484453,2.414062
3,6.799688,2.292969
4,6.563672,2.117188
5,6.312734,2.126953



  ✓ Done in 1648s
  train_loss   : 7.3325
  best_eval_loss: 2.1172
  steps run    : 1115
  [LR=0.0005_cosine_modules=[q,v,k,out]]  BLEU=28.20  chrF=52.04  ROUGE-1=62.08  METEOR=50.38  WER=65.09%

─────────────────────────────────────────────────────────────────
  PHASE 3 SUMMARY
─────────────────────────────────────────────────────────────────
  Label                Modules                             EvalLoss   BLEU     chrF    
  --------------------------------------------------------------
  LR=0.0005_cosine     ['q_proj', 'v_proj']                2.4961     24.21    45.79   
  LR=0.0005_cosine_mod ['q_proj', 'v_proj', 'k_proj', 'out_proj'] 2.1172     28.20    52.04   

  ✓ Best modules for final model: ['q_proj', 'v_proj', 'k_proj', 'out_proj']

  ── BEST OVERALL TRAINING CONFIG ─────────────────────────────
     lr             = 0.0005
     scheduler      = cosine
     target_modules = ['q_proj', 'v_proj', 'k_proj', 'out_proj']
  ─────────────────────────────────────────────────

In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAIN BEST CONFIG MODEL
# Train one more 5-epoch run with the best config found above.
# This model is used for the inference hyperparameter grid (beams + length_pen).
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "█"*65)
print("  TRAINING BEST CONFIG MODEL FOR INFERENCE GRID")
print("█"*65)

_, best_trained_model, best_eval_loss_final, _, _ = run_training_trial(
    trial_label    = f"BEST_lr{BEST_LR}_{BEST_SCHEDULER}_{'_'.join(BEST_MODULES)}",
    lr             = BEST_LR,
    scheduler_type = BEST_SCHEDULER,
    target_modules = BEST_MODULES,
)

print(f"\n  Best model ready. eval_loss={best_eval_loss_final:.4f}")



█████████████████████████████████████████████████████████████████
  TRAINING BEST CONFIG MODEL FOR INFERENCE GRID
█████████████████████████████████████████████████████████████████

  TRIAL: BEST_lr0.0005_cosine_q_proj_v_proj_k_proj_out_proj
  lr=0.0005  scheduler=cosine
  target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj']
  epochs=5  batch=16  grad_accum=2


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable params: 4,718,592  (0.761%)


Epoch,Training Loss,Validation Loss
1,8.514375,3.103516
2,7.499961,2.406250
3,6.797969,2.273438
4,6.579688,2.105469
5,6.326367,2.111328



  ✓ Done in 1668s
  train_loss   : 7.3400
  best_eval_loss: 2.1055
  steps run    : 1115

  Best model ready. eval_loss=2.1055


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE GRID — NUM_BEAMS x LENGTH_PENALTY
# No retraining. Tests all combinations on best_trained_model.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "█"*65)
print("  INFERENCE GRID: NUM_BEAMS x LENGTH_PENALTY")
print(f"  beams={INFERENCE_BEAM_OPTIONS}  length_penalties={INFERENCE_LENPEN_OPTIONS}")
print("█"*65)

inference_results = []

for nb_val in INFERENCE_BEAM_OPTIONS:
    for lp_val in INFERENCE_LENPEN_OPTIONS:
        label = f"beams={nb_val}_lp={lp_val}"
        print(f"\n  Testing {label}...")
        metrics = run_inference_eval(
            best_trained_model,
            num_beams      = nb_val,
            length_penalty = lp_val,
            label          = label,
        )
        inference_results.append(metrics)

# ── Inference grid summary ─────────────────────────────────────────────────────
print("\n" + "─"*65)
print("  INFERENCE GRID SUMMARY")
print("─"*65)
print(f"  {'Config':<22} {'BLEU':<8} {'chrF':<8} {'ROUGE-1':<9} {'METEOR':<9} {'WER%':<8} {'CER%':<8}")
print("  " + "-"*62)
for r in inference_results:
    print(f"  {r['label']:<22} {r['sacrebleu']:<8.2f} {r['chrf']:<8.2f} "
          f"{r['rouge1']:<9.2f} {r['meteor']:<9.2f} {r['wer']:<8.2f} {r['cer']:<8.2f}")

best_inf = max(inference_results, key=lambda x: x["sacrebleu"])
print(f"\n  ✓ Best inference config: {best_inf['label']}")
print(f"     BLEU={best_inf['sacrebleu']:.2f}  chrF={best_inf['chrf']:.2f}  METEOR={best_inf['meteor']:.2f}")



█████████████████████████████████████████████████████████████████
  INFERENCE GRID: NUM_BEAMS x LENGTH_PENALTY
  beams=[3, 5, 8]  length_penalties=[0.8, 1.0, 1.2]
█████████████████████████████████████████████████████████████████

  Testing beams=3_lp=0.8...
  [beams=3_lp=0.8]  BLEU=28.87  chrF=51.76  ROUGE-1=62.02  METEOR=50.61  WER=63.12%

  Testing beams=3_lp=1.0...
  [beams=3_lp=1.0]  BLEU=28.72  chrF=51.89  ROUGE-1=62.17  METEOR=50.72  WER=63.45%

  Testing beams=3_lp=1.2...
  [beams=3_lp=1.2]  BLEU=28.84  chrF=52.16  ROUGE-1=62.39  METEOR=51.05  WER=63.65%

  Testing beams=5_lp=0.8...
  [beams=5_lp=0.8]  BLEU=26.78  chrF=51.42  ROUGE-1=61.85  METEOR=50.03  WER=63.45%

  Testing beams=5_lp=1.0...
  [beams=5_lp=1.0]  BLEU=26.58  chrF=51.47  ROUGE-1=61.88  METEOR=49.85  WER=63.78%

  Testing beams=5_lp=1.2...
  [beams=5_lp=1.2]  BLEU=26.29  chrF=51.51  ROUGE-1=61.96  METEOR=50.02  WER=64.83%

  Testing beams=8_lp=0.8...
  [beams=8_lp=0.8]  BLEU=28.62  chrF=51.51  ROUGE-1=61.64  METE

## Using the Best Config for Your Final Training Run

After this notebook completes, take the **Recommended final config** printed above and paste it into your original notebook's config block:

```python
LEARNING_RATE        = <BEST_LR>
LR_SCHEDULER_TYPE    = '<BEST_SCHEDULER>'    # add to Seq2SeqTrainingArguments
LORA_TARGET_MODULES  = <BEST_MODULES>
NUM_EPOCHS           = 30                    # full run
NUM_BEAMS            = <BEST_BEAMS>
# In model.generate():  length_penalty=<BEST_LENGTH_PENALTY>
```

Results are also saved to `/kaggle/working/hp_tuning_results.json` for reference.


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# FULL HYPERPARAMETER TUNING SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd

print("\n" + "═"*70)
print("  HYPERPARAMETER TUNING — COMPLETE RESULTS")
print("═"*70)

# ── Training phase results ─────────────────────────────────────────────────────
df_hp = pd.DataFrame(HP_RESULTS)
cols_show = ["phase", "label", "lr", "scheduler", "modules",
             "best_eval_loss", "train_loss", "sacrebleu", "chrf",
             "rouge1", "rouge2", "rougeL", "meteor", "wer", "cer", "elapsed_s"]
cols_show = [c for c in cols_show if c in df_hp.columns]

print("\n  TRAINING TRIALS")
print(df_hp[cols_show].to_string(index=False))

# ── Inference grid results ──────────────────────────────────────────────────
df_inf = pd.DataFrame(inference_results)
print("\n\n  INFERENCE GRID (best trained model)")
print(df_inf.to_string(index=False))

# ── Winners ──────────────────────────────────────────────────────────────────
best_train = min(HP_RESULTS, key=lambda x: x["best_eval_loss"])
best_bleu_train = max(HP_RESULTS, key=lambda x: x["sacrebleu"])
best_bleu_inf   = max(inference_results, key=lambda x: x["sacrebleu"])

print("\n" + "═"*70)
print("  WINNERS")
print("═"*70)
print(f"  Lowest eval_loss : {best_train['label']}")
print(f"    lr={best_train['lr']}  scheduler={best_train['scheduler']}  modules={best_train['modules']}")
print(f"    eval_loss={best_train['best_eval_loss']:.4f}  BLEU={best_train['sacrebleu']:.2f}")
print()
print(f"  Highest BLEU (training trials)  : {best_bleu_train['label']}")
print(f"    BLEU={best_bleu_train['sacrebleu']:.2f}  chrF={best_bleu_train['chrf']:.2f}  METEOR={best_bleu_train['meteor']:.2f}")
print()
print(f"  Best inference config           : {best_bleu_inf['label']}")
print(f"    beams={best_bleu_inf['num_beams']}  length_penalty={best_bleu_inf['length_pen']}")
print(f"    BLEU={best_bleu_inf['sacrebleu']:.2f}  chrF={best_bleu_inf['chrf']:.2f}  METEOR={best_bleu_inf['meteor']:.2f}")
print()
print(f"  Recommended final config:")
print(f"    LEARNING_RATE        = {BEST_LR}")
print(f"    lr_scheduler_type    = '{BEST_SCHEDULER}'")
print(f"    LORA_TARGET_MODULES  = {BEST_MODULES}")
print(f"    NUM_BEAMS            = {best_bleu_inf['num_beams']}")
print(f"    length_penalty       = {best_bleu_inf['length_pen']}")
print("═"*70)

# ── Save all results to JSON ───────────────────────────────────────────────────
import json, os
results_path = "/kaggle/working/hp_tuning_results.json"
os.makedirs("/kaggle/working", exist_ok=True)
all_results = {
    "training_trials"   : HP_RESULTS,
    "inference_grid"    : inference_results,
    "best_train_config" : {
        "lr"            : BEST_LR,
        "scheduler"     : BEST_SCHEDULER,
        "modules"       : BEST_MODULES,
    },
    "best_inference_config": {
        "num_beams"     : best_bleu_inf["num_beams"],
        "length_penalty": best_bleu_inf["length_pen"],
    },
}
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\n  All results saved to: {results_path}")



══════════════════════════════════════════════════════════════════════
  HYPERPARAMETER TUNING — COMPLETE RESULTS
══════════════════════════════════════════════════════════════════════

  TRAINING TRIALS
 phase                                label     lr            scheduler                                    modules  best_eval_loss  train_loss  sacrebleu    chrf  rouge1  rouge2  rougeL  meteor     wer     cer  elapsed_s
     1                              LR=1e-4 0.0001               linear                       ['q_proj', 'v_proj']          3.3770      9.1779    11.3122 32.9085 42.1942 23.5318 40.7496 32.1584 84.1207 75.3459     1471.8
     1                   LR=3e-4 (baseline) 0.0003               linear                       ['q_proj', 'v_proj']          2.8281      8.3402    17.3261 40.5580 51.1739 31.7531 49.4091 39.6945 74.4094 61.0063     1492.2
     1                              LR=5e-4 0.0005               linear                       ['q_proj', 'v_proj']          2.5000  